In [1]:
import json
import os
import time
from tqdm.auto import tqdm
import pandas as pd
from openai import OpenAI
import minsearch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import numpy as np
from elasticsearch import Elasticsearch

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cargamos el modelo del Embedding
model_name = 'multi-qa-distilbert-cos-v1'
embedding_model = SentenceTransformer(model_name)

In [3]:
# Cargamos los datos de los archivos de las redes sociales
df = pd.read_json('../DATASET/tweets/quejas_sugerencias.json', lines=True)
df

,key,filename,text,tag
0,78bfd026-74a7-4a95-8999-5dea36aa0ad9,1.txt,"Reporte 30, día 30\nEl taller Popocatépetl Mot...",quejas
1,0cd3ce38-e20b-4822-b2a9-51f649f12e5b,10.txt,Cada cuánto tiempo salen los RTP para comunica...,quejas
2,a87f37c6-fa6a-403c-942c-6bfc8f537836,11.txt,"Caos en las inmediaciones del metro Chabacano,...",quejas
3,32e093e5-7a22-4008-afde-e0abe1220810,12.txt,Dicen que el servicio Atenea está de vuelta. ¿...,quejas
4,d7b6bf90-db3d-402c-a938-ad4a2f52e28e,14.txt,Uuy Ni Llega Ni Llegará a la Zona dónde Vivo E...,quejas
5,dd7398fb-f4a7-4e50-9a91-9a097ce04d90,16.txt,Hora y media esperando un camión de Cuajimalpa...,quejas
6,c41b5310-5d5e-41a8-a7ab-9ca6c36edd54,18.txt,"Al rato sacarán la flotilla Shakira, Paquita l...",quejas
7,e29ccbd0-2f21-465e-bae0-85ca236856ac,31.txt,Su servicio de la ruta 9C Puerta Grande - CC S...,quejas
8,27e5c5a3-9064-4ae2-b76a-f60c89bd8839,32.txt,LA RUTA 34-B QUE VIENE DE MIGUEL ANGEL DE QUEV...,quejas
9,2283fd78-11eb-42b7-af15-a2d45761398a,33.txt,Una vecina afectada denuncia a un sujeto acosa...,quejas


In [4]:
# Pasamos a diccionario
documents = df.to_dict(orient='records')
documents

[{'key': '78bfd026-74a7-4a95-8999-5dea36aa0ad9',
  'filename': '1.txt',
  'text': 'Reporte 30, día 30\nEl taller Popocatépetl Motors ( propiedad de \n@LuisMendozaBJ\n ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. \n@UCS_GCDMX\n \n@SSC_CDMX',
  'tag': 'quejas'},
 {'key': '0cd3ce38-e20b-4822-b2a9-51f649f12e5b',
  'filename': '10.txt',
  'text': 'Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) \n@RTP_CiudadDeMex\n \n@GobCDMX',
  'tag': 'quejas'},
 {'key': 'a87f37c6-fa6a-403c-942c-6bfc8f537836',
  'filename': '11.txt',
  'text': 'Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un corto circuito en el tra

In [5]:
# Indexamos los documentos, las columnas clave
index = minsearch.Index(
    text_fields=["text", "tag"],
    keyword_fields=["key"]
)

index.fit(documents)

In [6]:
# Funcion de busqueda, parametros de optimizacion para el boosting son las columnas clave,
# en este caso 'text' y 'tag'
def search(query):
    boost = {'text': 3.0, 'tag': 1.0}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict={},
        num_results=5
    )

    return results

In [7]:
# Generamos la plantilla del prompt
prompt_template = """
Vas a emular a un usuario que reportar una incidencia o reporte atraves de la aplicación APP CDMX.
En base al CONTENIDO infiere el tipo de REPORTE.

Registro:
CONTENIDO: {text}
REPORTE:
{tag}

""".strip()

In [8]:
# Probamos la funcion de busqueda, con el primer documento
res = search(documents[0]['text'])
res

[{'key': '57cccf51-ecb1-4e66-9a87-a7728875e0e6',
  'filename': '1.txt',
  'text': 'Reporte 30, día 30\nEl taller Popocatépetl Motors ( propiedad de \n@LuisMendozaBJ\n ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. \n@UCS_GCDMX\n \n@SSC_CDMX',
  'tag': 'sugerencias'},
 {'key': '78bfd026-74a7-4a95-8999-5dea36aa0ad9',
  'filename': '1.txt',
  'text': 'Reporte 30, día 30\nEl taller Popocatépetl Motors ( propiedad de \n@LuisMendozaBJ\n ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. \n@UCS_GCDMX\n \n@SSC_CDMX',
  'tag': 'quejas'},
 {'key': 'e7f149f2-7e33-4c13-b4e5-5cdc02e1f21f',
  'filename': '11.txt',
  'text': 'Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un cor

In [9]:
# Generamos el prompt
context = ""
for doc in res:
    context = context + f"key: {doc['key']}\filename: {doc['filename']}\text: {doc['text']}\text: {doc['tag']}\n\n"
    
print('ctx', context)
prompt = prompt_template.format(text=res[0]['text'], tag=res[0]['tag']).strip()

ctx key: 57cccf51-ecb1-4e66-9a87-a7728875e0e6ilename: 1.txt	ext: Reporte 30, día 30
El taller Popocatépetl Motors ( propiedad de 
@LuisMendozaBJ
 ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. 
@UCS_GCDMX
 
@SSC_CDMX	ext: sugerencias

key: 78bfd026-74a7-4a95-8999-5dea36aa0ad9ilename: 1.txt	ext: Reporte 30, día 30
El taller Popocatépetl Motors ( propiedad de 
@LuisMendozaBJ
 ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. 
@UCS_GCDMX
 
@SSC_CDMX	ext: quejas

key: e7f149f2-7e33-4c13-b4e5-5cdc02e1f21filename: 11.txt	ext: Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un corto circuito en el tramo de Jamaica a Pantitlán. 

Patrullas de la 
@SSC_CDMX
 y unidades 

# Levantando el servicio local para el modelo LLM

```sh
docker run -it -d \
    --rm \
    -v ollama:/root/.ollama \
    -p 11434:11434 \
    --name ollama \
    ollama/ollama

docker exec -it ollama ollama pull llama3.2:3b
```

In [10]:
# Nos conectamos a nuestro modelo con la libreria de OpenAI
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama', # API KEY por default
)

In [11]:
# Generamos la funcion del LLM
def llm(prompt, model="llama3.2:1b"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

In [21]:
# Generamos la funcion para el prompt
def build_prompt(query, search_results):
    prompt_template = """
    Vas a emular a un usuario que reporta una incidencia a través de la APP CDMX de la SUAC. 
    Tu tarea es analizar el texto del reporte y determinar si se trata de una "QUEJA" o de una "SUGERENCIA". 

    Para ello, ten en cuenta:

    - QUEJA: El reporte contiene descripciones de experiencias negativas, mal servicio, demoras u otras problemáticas.  
    - SUGERENCIA: El reporte ofrece propuestas de mejora, ideas, recomendaciones o extensiones de servicio.

    Analiza cuidadosamente el CONTENIDO a continuación y, basándote en estas pautas, responde únicamente con el tipo de reporte correspondiente sin explicaciones adicionales.

    Registro:
    CONTENIDO: {text}
    TIPO DE REPORTE: {tag}
    """.strip()

    context = ""
    for doc in search_results:
        context = context + f"key: {doc['key']}\filename: {doc['filename']}\text: {doc['text']}\text: {doc['tag']}\n\n"

    print('ctx', context)
    prompt = prompt_template.format(text=res[0]['text'], tag=res[0]['tag']).strip()
    return prompt

In [13]:
# Generamos la funcion del RAG
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    # print(prompt)
    answer = llm(prompt)
    return answer

In [19]:
queja_texto = "La ruta 1-A Metro Universidad - Ciudad Universitaria presenta pésimo servicio los fines de semana. Solo pasan 2 autobuses por hora cuando debería haber al menos uno cada 15 minutos. Los estudiantes y trabajadores de CU quedamos varados en Copilco esperando hasta 45 minutos. El sábado pasado perdí un examen por culpa de este mal servicio.\n@RTP_CiudadDeMex\n@UNAM_MX"
sugerencia_texto = "Propongo extender la ruta 5-B que va de Metro Coyoacán hasta el Centro Cultural Universitario, para que continúe hasta la Facultad de Ciencias y el Instituto de Investigaciones. Esto beneficiaría enormemente a la comunidad universitaria que debe caminar largas distancias o tomar otro transporte. También sugiero aumentar la frecuencia en horarios de entrada y salida de clases (7-9 AM y 6-8 PM).\n@RTP_CiudadDeMex\n@AlcCoyoacan"

### Prueba

In [23]:
search(queja_texto)

[{'key': '40677088-f3b0-47d0-89f5-ae1ae5877048',
  'filename': '10.txt',
  'text': 'Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) \n@RTP_CiudadDeMex\n \n@GobCDMX',
  'tag': 'sugerencias'},
 {'key': '0cd3ce38-e20b-4822-b2a9-51f649f12e5b',
  'filename': '10.txt',
  'text': 'Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) \n@RTP_CiudadDeMex\n \n@GobCDMX',
  'tag': 'quejas'},
 {'key': '3124e2ed-edff-40af-ac6f-1cfc57b78626',
  'filename': '12.txt',
  'text': 'Dicen que el servicio Atenea está de vuelta. ¿Es cierto \n@RTP_CiudadDeMex\n? \n\nNo es lo mejor en rutas con 5 autobuses y frecuencias de paso de 55 min

In [22]:
# Test RAG
rag(queja_texto)

ctx key: 40677088-f3b0-47d0-89f5-ae1ae5877048ilename: 10.txt	ext: Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) 
@RTP_CiudadDeMex
 
@GobCDMX	ext: sugerencias

key: 0cd3ce38-e20b-4822-b2a9-51f649f12e5bilename: 10.txt	ext: Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) 
@RTP_CiudadDeMex
 
@GobCDMX	ext: quejas

key: 3124e2ed-edff-40af-ac6f-1cfc57b78626ilename: 12.txt	ext: Dicen que el servicio Atenea está de vuelta. ¿Es cierto 
@RTP_CiudadDeMex
? 

No es lo mejor en rutas con 5 autobuses y frecuencias de paso de 55 minutos.
Pero bueno, capital de la transformación. 

en RTP 162D: Santa Catarina - Metro U

'SUGERENCIA'

In [24]:
rag(sugerencia_texto)

ctx key: e7f149f2-7e33-4c13-b4e5-5cdc02e1f21filename: 11.txt	ext: Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un corto circuito en el tramo de Jamaica a Pantitlán. 

Patrullas de la 
@SSC_CDMX
 y unidades de 
@RTP_CiudadDeMex
, apoyan en el traslado de personas. 	ext: sugerencias

key: a87f37c6-fa6a-403c-942c-6bfc8f537836ilename: 11.txt	ext: Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un corto circuito en el tramo de Jamaica a Pantitlán. 

Patrullas de la 
@SSC_CDMX
 y unidades de 
@RTP_CiudadDeMex
, apoyan en el traslado de personas. 	ext: quejas

key: d8216e69-d08e-4a96-8ce7-802939da5624ilename: 32.txt	ext: LA RUTA 34-B QUE VIENE DE MIGUEL ANGEL DE QUEVEDO A SANTA FE, ES UNA VERDADERA PESADILLA TODOS LOS DIAS A CUALQUIER HORA, MAS DE UNA HORA ESPERANDO UN ECOBUS, CON FILAS ENORMES Y CUANDO PASA UN CAMION OBVIAMENTE NO HACE PARADA POR LO ATAS

'SUGERENCIA'

# Embedding

In [25]:
# convertimos el texto de la queja a un vector
v = embedding_model.encode(queja_texto)
v

array([ 4.43144701e-02, -6.50726333e-02,  6.79378733e-02,  4.69318405e-02,
        2.80194836e-05,  2.16785651e-02,  1.43156173e-02,  7.23251849e-02,
       -9.97945899e-04,  1.46444626e-02, -2.62833908e-02,  3.37721258e-02,
        1.18604796e-02, -1.91324018e-02,  3.20419148e-02, -3.30612473e-02,
        2.71816598e-03,  1.35412905e-02, -1.21117961e-02, -6.57607894e-03,
       -1.36745935e-02,  4.00083661e-02,  6.19315915e-02, -2.34391820e-02,
        9.59563628e-03, -2.83844639e-02, -2.87872162e-02,  2.56584939e-02,
       -3.46180908e-02,  3.31979915e-02,  1.43917613e-02,  1.77917834e-02,
       -6.30930113e-03,  6.09504320e-02,  4.95030917e-02,  2.45566461e-02,
       -1.33030750e-02, -2.58404501e-02,  1.05184168e-02,  3.06843338e-03,
       -3.07629444e-03,  2.18579313e-03, -2.18125675e-02, -2.73886900e-02,
        1.77968331e-02, -3.14272828e-02,  5.93590401e-02, -2.63804384e-02,
       -6.11698767e-03,  4.34482656e-02, -8.48135725e-03,  3.63481790e-02,
        3.14217433e-02,  

In [26]:
# Lo convertimos a np.array
X = np.array(v)
X.shape # Tenemos un modelo que es un vector de 768 dimensiones

(768,)

In [27]:
# Convertimos nuestros documentos a vectores
embeddings = []
embedded_documents = []

for doc in tqdm(documents):
    text = doc['text']
    tag = doc['tag']
    tag_text = f'{text} {tag}'
    doc["tag_text"] = embedding_model.encode(tag_text)
    embeddings.append(doc["tag_text"])
    embedded_documents.append(doc)

  0%|          | 0/40 [00:00<?, ?it/s]

In [28]:
X = np.array(embeddings)
X.shape

(40, 768)

In [29]:
# Aplicamos el producto punto, usando nuestro vector de la queja
results = []
for emb in tqdm(embeddings):
    results.append(emb.dot(v))

  0%|          | 0/40 [00:00<?, ?it/s]

In [30]:
max(results) # Tenemos el maximo de la similitud de 0.71

np.float32(0.72876996)

In [31]:
# Generamos la clase VectorSearchEngine para buscar en el embedding
class VectorSearchEngine():
    def __init__(self, documents, embeddings):
        self.documents = documents
        self.embeddings = embeddings

    def search(self, v_query, num_results=10):
        scores = self.embeddings.dot(v_query)
        idx = np.argsort(-scores)[:num_results]
        return [self.documents[i] for i in idx]

# Generamos el objeto de la clase
search_engine = VectorSearchEngine(documents=documents, embeddings=X)
# Buscamos
search_engine.search(v, num_results=5)

[{'key': '0cd3ce38-e20b-4822-b2a9-51f649f12e5b',
  'filename': '10.txt',
  'text': 'Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) \n@RTP_CiudadDeMex\n \n@GobCDMX',
  'tag': 'quejas',
  'tag_text': array([ 1.74262356e-02, -6.78111687e-02,  4.49705049e-02,  5.68379350e-02,
          2.47347467e-02,  1.54437162e-02, -4.11325656e-02,  8.93601179e-02,
          1.58308819e-02, -9.16025601e-03,  2.82311602e-03,  2.44602021e-02,
          4.51303041e-03, -3.48726027e-02,  2.39740126e-02, -1.74260698e-02,
         -1.75881553e-02,  1.21584302e-02, -1.30273178e-02, -9.63333063e-03,
         -5.94662502e-02,  5.91551978e-03,  2.64637712e-02, -1.72877721e-02,
         -1.22449445e-02,  1.41687226e-04, -6.90841465e-04,  4.29235883e-02,
         -4.67407331e-02,  1.28435055e-02,  1.55877508e-03,  1.57393571

> De los resultados el que da 0.71 es una queja :D

# Integrando ElasticSearch


Levantamos Elasticsearch

```sh
docker run -it \
    --rm \
    --name elasticsearch \
    -p 9200:9200 \
    -p 9300:9300 \
    -e "discovery.type=single-node" \
    -e "xpack.security.enabled=false" \
    -e "ES_JAVA_OPTS=-Xms512m -Xmx512m" \
    elasticsearch:8.4.3
```

In [32]:
# Nos conectamos a Elasticsearch
es_client = Elasticsearch('http://localhost:9200')

print(es_client.info())

{'name': 'aa6df578cebf', 'cluster_name': 'docker-cluster', 'cluster_uuid': '5h8inkG0T8Otk1jC4cVfQw', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [43]:
# Creamos la configuración del índice
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "filename": {"type": "text"},
            "tag": {"type": "text"},
            "tag_text": {"type": "dense_vector", "dims": 768, "index": True, "similarity": "cosine"},  # tomando los 768 tokens
            "key": {"type": "keyword"} 
        }
    }
}

index_name = "reportes_suac"

es_client.indices.delete(index=index_name, ignore=[400, 404])  # Elimina el índice si ya existe
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'reportes_suac'})

In [44]:
# Agregamos los documentos al índice
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/40 [00:00<?, ?it/s]

In [45]:
# Crear una consulta de búsqueda
query = {
    "field": "tag_text",
    "query_vector": v,
    "k": 5,
    "num_candidates": 10000, 
}

In [47]:
res = es_client.search(index=index_name, knn=query, source=["text", "tag", "key", "tag_text"])
res["hits"]["hits"]

[{'_index': 'reportes_suac',
  '_id': 'ALZbCZcBHEC9CGZLhqvM',
  '_score': 0.86438483,
  '_source': {'tag_text': [0.01742623560130596,
    -0.06781116873025894,
    0.04497050493955612,
    0.056837935000658035,
    0.024734746664762497,
    0.015443716198205948,
    -0.04113256558775902,
    0.08936011791229248,
    0.015830881893634796,
    -0.009160256013274193,
    0.002823116024956107,
    0.0244602020829916,
    0.004513030406087637,
    -0.03487260267138481,
    0.023974012583494186,
    -0.01742606982588768,
    -0.01758815534412861,
    0.012158430181443691,
    -0.013027317821979523,
    -0.009633330628275871,
    -0.059466250240802765,
    0.005915519781410694,
    0.026463771238923073,
    -0.01728777214884758,
    -0.012244944460690022,
    0.00014168722555041313,
    -0.0006908414652571082,
    0.042923588305711746,
    -0.046740733087062836,
    0.01284350547939539,
    0.001558775082230568,
    0.015739357098937035,
    -0.0011521739652380347,
    0.053310152143239975,
 

In [35]:
# Creamos la funcion de busqueda en Elasticsearch
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["text", "tag^3"],
                        "type": "best_fields"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [36]:
# Integramos la busqueda de Elasticsearch con RAG
def rag_elastic(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [38]:
rag_elastic(sugerencia_texto)

ctx key: 32e093e5-7a22-4008-afde-e0abe1220810ilename: 12.txt	ext: Dicen que el servicio Atenea está de vuelta. ¿Es cierto 
@RTP_CiudadDeMex
? 

No es lo mejor en rutas con 5 autobuses y frecuencias de paso de 55 minutos.
Pero bueno, capital de la transformación. 

en RTP 162D: Santa Catarina - Metro Universidad en Coyoacán, Tlalpan, Tláhuac, CDMX	ext: quejas

key: 3124e2ed-edff-40af-ac6f-1cfc57b78626ilename: 12.txt	ext: Dicen que el servicio Atenea está de vuelta. ¿Es cierto 
@RTP_CiudadDeMex
? 

No es lo mejor en rutas con 5 autobuses y frecuencias de paso de 55 minutos.
Pero bueno, capital de la transformación. 

en RTP 162D: Santa Catarina - Metro Universidad en Coyoacán, Tlalpan, Tláhuac, CDMX	ext: sugerencias

key: e29ccbd0-2f21-465e-bae0-85ca236856acilename: 31.txt	ext: Su servicio de la ruta 9C Puerta Grande - CC Santa Fe esta sobrepasado. Filas de hasta 200 personas esperando RTP a las 6 am. Han llegado hasta 3 Camiones y los 3 se super llenan. Y aun asi sigue llegando mucha

'SUGERENCIA'

# Metrica: MMR & Hit-rate

In [68]:
def hit_rate(relevance_total):
    cnt = 0
    try:
        relevance_total = np.array(relevance_total)
        for line in relevance_total:
            if True in line:
                cnt = cnt + 1

        return cnt / len(relevance_total)
    except Exception as e:
        print(f"Error calculating hit rate: {e}")
        return 0.0

def mrr(relevance_total):
    total_score = 0.0
    try:
        for line in relevance_total:
            for rank in range(len(line)):
                if line[rank] == True:
                    total_score = total_score + 1 / (rank + 1)

        return total_score / len(relevance_total)
    except Exception as e:
        print(f"Error calculating hit rate: {e}")
        return 0.0

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['key']
        results = search_function(q)
        relevance = [d['key'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [72]:
# Importamos el ground truth

df_ground_truth = pd.read_json('../DATASET/tweets/sample_ground_of_truth.json', lines=True)
ground_truth = df_ground_truth.to_dict(orient='records')
ground_truth

[{'key': '78bfd026-74a7-4a95-8999-5dea36aa0ad9',
  'filename': '1.txt',
  'text': 'Reporte 30, día 30\nEl taller Popocatépetl Motors ( propiedad de \n@LuisMendozaBJ\n ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. \n@UCS_GCDMX\n \n@SSC_CDMX',
  'tag': 'quejas'},
 {'key': '0cd3ce38-e20b-4822-b2a9-51f649f12e5b',
  'filename': '10.txt',
  'text': 'Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) \n@RTP_CiudadDeMex\n \n@GobCDMX',
  'tag': 'quejas'},
 {'key': 'a87f37c6-fa6a-403c-942c-6bfc8f537836',
  'filename': '11.txt',
  'text': 'Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un corto circuito en el tra

In [50]:
def numpy_cosine_search(q):
    text = q['text']

    v_q = embedding_model.encode(text)

    return search_engine.search(v_q, num_results=5)

In [73]:
evaluate(ground_truth, numpy_cosine_search)

  0%|          | 0/6 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.6666666666666666}

In [63]:
def elastic_search_knn(field, vector):
    knn = {
        "field": field,
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000
    }

    search_query = {
        "knn": knn,
        "_source": ["text", "tag", "tag)text", "key"]
    }

    es_results = es_client.search(
        index=index_name,
        body=search_query
    )
    
    result_docs = []
    
    for hit in es_results['hits']['hits']:
        result_docs.append(hit['_source'])

    return result_docs

In [64]:
def question_vector_knn(q):
    question = q['text']

    v_q = embedding_model.encode(question)

    return elastic_search_knn('tag_text', v_q)

In [74]:
question_vector_knn(ground_truth[5])

[{'text': 'Pues este señor, increpó al chófer del RTP. No sé quería mover ni dar paso al bus público.\nOjalá \n@SSC_CDMX\n y las fotomultas hagan lo suyo.\nEs el "pan nuestro de cada día" en el carril reversible,  desde eje 6 hasta La Merced. \nAhí les encargo \n@UCS_GCDMX\n sus polis no trabajan.',
  'tag': 'quejas',
  'key': '3228e90f-5bcb-42f3-bd04-7fe037e23dbb'},
 {'text': 'Pues este señor, increpó al chófer del RTP. No sé quería mover ni dar paso al bus público.\nOjalá \n@SSC_CDMX\n y las fotomultas hagan lo suyo.\nEs el "pan nuestro de cada día" en el carril reversible,  desde eje 6 hasta La Merced. \nAhí les encargo \n@UCS_GCDMX\n sus polis no trabajan.',
  'tag': 'sugerencias',
  'key': 'bc9005cf-6055-4f70-8fec-c9370f6f8eb8'},
 {'text': 'Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad? 30 minutos a 1 hora? Solo es simulación para decir que dan servicio? (19.3248099, -99.1733275) \n@

In [76]:
evaluate(ground_truth, question_vector_knn)

  0%|          | 0/6 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.6666666666666666}

## RAG

In [81]:
def build_prompt_categorize(query):
    prompt_template = """
    Eres un clasificador de textos especializado en identificar si un mensaje corresponde a una queja o una sugerencia relacionada con servicios de transporte público.
    
    Definiciones:
    - QUEJA: Texto que reporta un problema específico, denuncia una situación negativa, describe un incidente concreto, o expresa inconformidad por un servicio deficiente. Incluye:

        - Reportes de accidentes o situaciones peligrosas
        - Denuncias de mal uso de espacios públicos
        - Descripción de problemas operativos específicos
        - Situaciones de caos o emergencia

    - SUGERENCIA: Texto que propone mejoras, solicita información sobre servicios, expresa necesidades de manera constructiva, o busca soluciones. Incluye:

        - Solicitudes de mejor frecuencia de servicio
        - Peticiones de intervención para mejorar situaciones
        - Preguntas sobre horarios o rutas
        - Propuestas de optimización

    Ejemplos de referencia:
    - QUEJAS:

        - "El taller Popocatépetl Motors usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales."
        - "Caos en las inmediaciones del metro Chabacano, por suspensión en el servicio de la línea 9 del #MetroCDMX debido a un corto circuito"

    - SUGERENCIAS:

        - "Cada cuánto tiempo salen los RTP para comunicar el surponiente de la ciudad sin buen servicio de transporte concesionado desde metro universidad?"
        - "No hay camiones en Acoxpa, ya tardaron demasiado, gente a pleno sol, pésimo servicio ruta 300-A"

    Formato de respuesta:
    Clasifica el siguiente texto y responde únicamente:

    TIPO: [queja/sugerencia]
    JUSTIFICACIÓN: [Breve explicación de por qué pertenece a esa categoría]

    Texto a clasificar:
    {solicitud}""".strip()

    #context = ""
    #for doc in search_results:
    #    context = context + entry_template.format(**doc) + "\n\n"

    return prompt_template.format(solicitud=query).strip()

In [83]:
# Integramos la busqueda de Elasticsearch con RAG
def rag_llm(query):
    prompt = build_prompt_categorize(query)
    answer = llm(prompt)
    return answer

In [84]:
rag_llm(ground_truth[0]['text'])

'Clasificación: QUEJAS\n\nJustificación: El texto describe un problema específico relacionado con el uso del carril del RTP como estacionamiento y extensión de su patio de maniobras, lo que indica una queja sobre la situación.'

In [85]:
ground_truth[0]

{'key': '78bfd026-74a7-4a95-8999-5dea36aa0ad9',
 'filename': '1.txt',
 'text': 'Reporte 30, día 30\nEl taller Popocatépetl Motors ( propiedad de \n@LuisMendozaBJ\n ) usa el carril del RTP como estacionamiento y extensión de su patio de maniobras, bloquea banquetas y pasos peatonales. Debido a esto es el causante de varios accidentes viales. \n@UCS_GCDMX\n \n@SSC_CDMX',
 'tag': 'quejas'}